# Streaming data processing

Owner: Oscar. Live Wikipedia changes (Wikimedia EventStreams) are sent to Kafka by `streaming/wikistream_producer.py`, then this notebook reads the Kafka topic with Spark Structured Streaming and keeps the changes made on Wikipedia pages of IMDb films, series and people (silver layer).

How to run (see PROJECT.md):

1. `docker compose up -d` (Kafka + Jupyter);
2. run this notebook down to the cell **Start the streaming queries**;
3. in a terminal on the host: `python streaming/wikistream_producer.py --minutes 25`;
4. come back here: the notebook waits `RUN_MINUTES` then stops the queries and shows the results.

In [1]:
import sys
sys.path.insert(0, "../jobs")  # notebooks/ -> jobs/

import json
import shutil
import time

import pandas as pd
from pyspark.sql import DataFrame, Window, functions as F, types as T

from common import DATA_DIR, PROJECT_ROOT, get_spark, read_silver

# with_kafka downloads the spark-sql-kafka connector the first time
spark = get_spark("imdb_streaming", driver_memory="2g", with_kafka=True, cores=2)
spark.sparkContext.setLogLevel("WARN")
spark.conf.set("spark.sql.shuffle.partitions", "4")  # small streaming state: few partitions

STREAM_DIR = DATA_DIR / "streaming"
RESULTS_DIR = PROJECT_ROOT / "results" / "streaming"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## Parameters

In [2]:
KAFKA_BOOTSTRAP = "kafka:19092"   # Kafka listener inside the Docker network
TOPIC = "wiki-changes"
WIKIS = ["en.wikipedia.org"]      # wikis kept (the IMDb titles are mostly in English)
CHANGE_TYPES = ["edit", "new"]    # page edits and page creations
NAMESPACE_ARTICLE = 0             # namespace 0 = articles (no talk / user / template pages)

WATERMARK = "10 minutes"          # how late an event may arrive and still be counted
WINDOW = "5 minutes"              # tumbling window for the aggregations
TRIGGER = "30 seconds"            # micro-batch interval
RUN_MINUTES = 25                  # duration of the test window (15 to 30 min)

MIN_VOTES_QUALIFIED = 1_000       # "X (film)", "X (2010 film)", "X (TV series)": known titles
MIN_VOTES_BARE = 50_000           # bare page title "X": only very popular titles
MIN_VOTES_PERSON = 20_000         # people credited on titles with at least this many votes
PERSON_CATEGORIES = ["actor", "actress", "director", "writer", "producer", "composer"]

RESET = True                      # True: start from an empty checkpoint / output

## 1. Design and implement a PySpark Structured Streaming pipeline that ingests live Wikipedia change events via Kafka and identifies updates relevant to your IMDb dataset.

### 1.1 Matching keys

Wikipedia page titles and IMDb names are turned into the same normalized key (lower case, letters and digits only). Wikipedia uses a qualifier in parentheses to separate homonyms, e.g. `Oppenheimer (film)`, `Dune (2021 film)`, `The Office (American TV series)`, `Michael Jordan (actor)`. The qualifier gives the kind of entity and sometimes the year, which makes the match much more precise.

In [3]:
def norm(col):
    """Normalized matching key: lower case, letters and digits, single spaces."""
    return F.trim(F.regexp_replace(F.lower(col), r"[^\p{L}\p{N}]+", " "))


QUALIFIER_KIND = (
    ("film", r"\bfilm\b|\bmovie\b"),
    ("series", r"series|sitcom|\bshow\b|tv program|miniseries"),
    ("person", r"actor|actress|director|filmmaker|screenwriter|writer|producer|comedian|composer"),
)


def wiki_keys(title_col):
    """Candidate lookup keys for a Wikipedia page title: kind|year|normalized name."""
    base = norm(F.regexp_replace(title_col, r"\s*\([^)]*\)\s*$", ""))
    qualifier = F.lower(F.regexp_extract(title_col, r"\(([^)]*)\)\s*$", 1))
    year = F.regexp_extract(qualifier, r"((?:18|19|20)\d{2})", 1)

    kind = F.when(qualifier == "", F.lit("none"))
    for name, pattern in QUALIFIER_KIND:
        kind = kind.when(qualifier.rlike(pattern), F.lit(name))
    kind = kind.otherwise(F.lit("other"))  # e.g. "Paris (band)": not an IMDb entity

    return (
        F.when(kind == "none", F.array(F.concat_ws("|", F.lit("any"), F.lit(""), base),
                                       F.concat_ws("|", F.lit("person"), F.lit(""), base)))
        .when(kind == "person", F.array(F.concat_ws("|", kind, F.lit(""), base)))
        .when(kind.isin("film", "series"), F.array(F.concat_ws("|", kind, year, base)))
        .otherwise(F.array().cast("array<string>"))
    )

### 1.2 Static lookup table built from the silver layer

One row per key, with the IMDb entity it points to. When several IMDb entities share a key (remakes, homonyms) the one with the most votes is kept.

In [4]:
LOOKUP_PATH = STREAM_DIR / "imdb_lookup"

titles = (
    read_silver(spark, "title.basics")
    .join(read_silver(spark, "title.ratings"), "tconst")
    .withColumn("kind", F.when(F.col("titleType").isin("movie", "tvMovie"), "film")
                         .when(F.col("titleType").isin("tvSeries", "tvMiniSeries"), "series"))
    .filter(F.col("kind").isNotNull() & (F.col("numVotes") >= MIN_VOTES_QUALIFIED))
    .select(F.col("tconst").alias("imdb_id"), F.col("primaryTitle").alias("imdb_name"),
            "kind", F.col("titleType").alias("imdb_type"), F.col("startYear").alias("imdb_year"),
            "averageRating", "numVotes", norm(F.col("primaryTitle")).alias("name_key"))
)

title_keys = (
    titles.withColumn("lookup_key", F.concat_ws("|", "kind", F.col("imdb_year").cast("string"), "name_key"))
    .unionByName(titles.withColumn("lookup_key", F.concat_ws("|", "kind", F.lit(""), "name_key")))
    .unionByName(titles.filter(F.col("numVotes") >= MIN_VOTES_BARE)
                 .withColumn("lookup_key", F.concat_ws("|", F.lit("any"), F.lit(""), "name_key")))
    .withColumn("entity_type", F.lit("title"))
)

# people: credited (actor, director...) on at least one popular title
popular = read_silver(spark, "title.ratings").filter(F.col("numVotes") >= MIN_VOTES_PERSON)
people = (
    read_silver(spark, "title.principals")
    .filter(F.col("category").isin(PERSON_CATEGORIES))
    .join(F.broadcast(popular.select("tconst", "numVotes")), "tconst")  # map-only scan of principals
    .groupBy("nconst").agg(F.max("numVotes").alias("numVotes"))
    .join(read_silver(spark, "name.basics"), "nconst")
    .select(F.col("nconst").alias("imdb_id"), F.col("primaryName").alias("imdb_name"),
            F.lit("person").alias("kind"), F.lit("person").alias("imdb_type"),
            F.col("birthYear").alias("imdb_year"), F.lit(None).cast("double").alias("averageRating"),
            "numVotes", norm(F.col("primaryName")).alias("name_key"))
    .withColumn("lookup_key", F.concat_ws("|", F.lit("person"), F.lit(""), "name_key"))
    .withColumn("entity_type", F.lit("person"))
)

best = Window.partitionBy("lookup_key").orderBy(F.desc("numVotes"), "imdb_id")
lookup = (
    title_keys.unionByName(people)
    .filter(F.col("name_key") != "")
    .withColumn("rank", F.row_number().over(best))
    .filter("rank = 1")
    .drop("rank", "kind", "name_key")
)

if RESET or not (LOOKUP_PATH / "_SUCCESS").exists():
    lookup.write.mode("overwrite").parquet(str(LOOKUP_PATH))
lookup = spark.read.parquet(str(LOOKUP_PATH)).cache()

lookup.groupBy(F.split("lookup_key", "\\|")[0].alias("key_kind"), "entity_type") \
    .count().orderBy("key_kind").show()

+--------+-----------+------+
|key_kind|entity_type| count|
+--------+-----------+------+
|     any|      title|  5235|
|    film|      title|101068|
|  person|     person| 64316|
|  series|      title| 25449|
+--------+-----------+------+



Check of the matching rules on a few page titles, before using them on the stream:

In [5]:
examples = spark.createDataFrame([(t,) for t in [
    "Inception", "Oppenheimer (film)", "Dune (2021 film)", "Dune (1984 film)", "Breaking Bad",
    "The Office (American TV series)", "Tom Hanks", "Paris", "Paris (band)", "Talk:Inception",
]], "title string")

(examples.withColumn("lookup_key", F.explode_outer(wiki_keys(F.col("title"))))
 .join(F.broadcast(lookup), "lookup_key", "left")
 .select("title", "lookup_key", "entity_type", "imdb_id", "imdb_name", "imdb_year", "numVotes")
 .show(truncate=False))

+-------------------------------+----------------------+-----------+----------+------------+---------+--------+
|title                          |lookup_key            |entity_type|imdb_id   |imdb_name   |imdb_year|numVotes|
+-------------------------------+----------------------+-----------+----------+------------+---------+--------+
|Inception                      |any||inception        |title      |tt1375666 |Inception   |2010     |2888117 |
|Inception                      |person||inception     |NULL       |NULL      |NULL        |NULL     |NULL    |
|Oppenheimer (film)             |film||oppenheimer     |title      |tt15398776|Oppenheimer |2023     |1111504 |
|Dune (2021 film)               |film|2021|dune        |title      |tt1160419 |Dune        |2021     |1101446 |
|Dune (1984 film)               |film|1984|dune        |title      |tt0087182 |Dune        |1984     |194943  |
|Breaking Bad                   |any||breaking bad     |title      |tt0903747 |Breaking Bad|2008     |27

### 1.3 Read the Kafka topic and parse the Wikimedia events

The event time is the time of the change on Wikipedia (`timestamp`, in seconds), not the time the message reached Kafka. The difference between the two is kept as `delay_seconds`.

In [6]:
EVENT_SCHEMA = T.StructType([
    T.StructField("meta", T.StructType([
        T.StructField("id", T.StringType()),
        T.StructField("dt", T.StringType()),
        T.StructField("uri", T.StringType()),
    ])),
    T.StructField("id", T.LongType()),
    T.StructField("type", T.StringType()),
    T.StructField("namespace", T.IntegerType()),
    T.StructField("title", T.StringType()),
    T.StructField("comment", T.StringType()),
    T.StructField("timestamp", T.LongType()),
    T.StructField("user", T.StringType()),
    T.StructField("bot", T.BooleanType()),
    T.StructField("minor", T.BooleanType()),
    T.StructField("server_name", T.StringType()),
    T.StructField("length", T.StructType([
        T.StructField("old", T.LongType()),
        T.StructField("new", T.LongType()),
    ])),
    T.StructField("simulated_delay_s", T.IntegerType()),  # added by our producer (late data test)
])


def parse_events(raw: DataFrame) -> DataFrame:
    """raw: columns value (json string) and kafka_time (timestamp)."""
    return (
        raw.select(F.from_json("value", EVENT_SCHEMA).alias("e"), "kafka_time")
        .select("e.*", "kafka_time")
        .withColumn("event_time", F.timestamp_seconds("timestamp"))
        .withColumn("delay_seconds", F.col("kafka_time").cast("long") - F.col("timestamp"))
        .withColumn("bytes_changed", F.coalesce(F.col("length.new"), F.lit(0)) - F.coalesce(F.col("length.old"), F.lit(0)))
        .select(F.col("meta.id").alias("event_id"), "event_time", "kafka_time", "delay_seconds",
                "simulated_delay_s", "server_name", "type", "namespace", "title", "user", "bot",
                "minor", "bytes_changed", "comment", F.col("meta.uri").alias("page_url"))
    )


def relevant_changes(events: DataFrame) -> DataFrame:
    """Filter the stream and keep the changes on pages of IMDb entities."""
    return (
        events
        .filter(F.col("server_name").isin(WIKIS) & F.col("type").isin(CHANGE_TYPES)
                & (F.col("namespace") == NAMESPACE_ARTICLE) & F.col("event_time").isNotNull())
        .withWatermark("event_time", WATERMARK)
        .dropDuplicatesWithinWatermark(["event_id"])  # the producer may resend a message
        .withColumn("lookup_key", F.explode(wiki_keys(F.col("title"))))
        .join(F.broadcast(lookup), "lookup_key")      # stream-static join
        .drop("lookup_key")
    )


kafka_stream = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .option("failOnDataLoss", "false")
    .load()
    .select(F.col("value").cast("string").alias("value"), F.col("timestamp").alias("kafka_time"))
)

matches = relevant_changes(parse_events(kafka_stream))
matches.printSchema()

root
 |-- event_id: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- kafka_time: timestamp (nullable = true)
 |-- delay_seconds: long (nullable = true)
 |-- simulated_delay_s: integer (nullable = true)
 |-- server_name: string (nullable = true)
 |-- type: string (nullable = true)
 |-- namespace: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- user: string (nullable = true)
 |-- bot: boolean (nullable = true)
 |-- minor: boolean (nullable = true)
 |-- bytes_changed: long (nullable = false)
 |-- comment: string (nullable = true)
 |-- page_url: string (nullable = true)
 |-- imdb_id: string (nullable = true)
 |-- imdb_name: string (nullable = true)
 |-- imdb_type: string (nullable = true)
 |-- imdb_year: integer (nullable = true)
 |-- averageRating: double (nullable = true)
 |-- numVotes: integer (nullable = true)
 |-- entity_type: string (nullable = true)



### 1.4 Aggregation by event-time window

Number of changes, distinct editors and bytes changed per IMDb entity and per 5-minute window. With the watermark, a window is final once the watermark has passed its end, and events arriving later than the watermark are dropped (`numRowsDroppedByWatermark` in the metrics).

In [7]:
window_counts = (
    matches
    .groupBy(F.window("event_time", WINDOW).alias("window"), "entity_type", "imdb_id", "imdb_name")
    .agg(F.count("*").alias("changes"),
         F.approx_count_distinct("user").alias("editors"),
         F.sum("bytes_changed").alias("bytes_changed"))
    .select(F.col("window.start").alias("window_start"), F.col("window.end").alias("window_end"),
            "entity_type", "imdb_id", "imdb_name", "changes", "editors", "bytes_changed")
)

### Start the streaming queries

In [8]:
MATCHES_PATH = STREAM_DIR / "matches"
WINDOWS_PATH = STREAM_DIR / "window_counts"
CHECKPOINTS = STREAM_DIR / "_checkpoints"

if RESET:
    for path in (MATCHES_PATH, WINDOWS_PATH, CHECKPOINTS):
        shutil.rmtree(path, ignore_errors=True)


def save_window_batch(batch: DataFrame, batch_id: int):
    """update mode: each micro-batch holds the windows that changed; we keep every version."""
    batch.withColumn("batch_id", F.lit(batch_id)).write.mode("append").parquet(str(WINDOWS_PATH))


q_matches = (
    matches.writeStream.queryName("matches")
    .format("parquet").option("path", str(MATCHES_PATH))
    .option("checkpointLocation", str(CHECKPOINTS / "matches"))
    .outputMode("append").trigger(processingTime=TRIGGER).start()
)
q_windows = (
    window_counts.writeStream.queryName("window_counts")
    .foreachBatch(save_window_batch)
    .option("checkpointLocation", str(CHECKPOINTS / "window_counts"))
    .outputMode("update").trigger(processingTime=TRIGGER).start()
)
print("queries started: now run the producer on the host")

queries started: now run the producer on the host


In [ ]:
# wait for the test window, printing a short status every minute
total = 0
for minute in range(1, RUN_MINUTES + 1):
    time.sleep(60)
    for q in (q_matches, q_windows):
        if not q.isActive:  # a query that crashed: show why instead of waiting
            raise RuntimeError(f"query {q.name} stopped: {q.exception()}")
    total = sum(p["numInputRows"] for p in q_matches.recentProgress)
    w = q_windows.lastProgress or {}
    dropped = sum(s.get("numRowsDroppedByWatermark", 0)
                  for p in q_windows.recentProgress for s in p.get("stateOperators", []))
    print(f"{minute:>2} min | Kafka messages read {total:>6} | {q_matches.status['message']:<40} "
          f"| watermark {w.get('eventTime', {}).get('watermark', '-')} | dropped (late) {dropped}")
    if minute == 3 and total == 0:
        print("   -> nothing received from Kafka yet: is the producer running? is the kafka container up?")

for q in (q_matches, q_windows):
    q.stop()


## 2. You may define your own strategy for filtering or sub setting the incoming stream. However, your filtering criteria must be sufficiently broad to guarantee that multiple matching events are captured and processed within a 15 to 30 minute test window.

In [ ]:
if not list(MATCHES_PATH.glob("*.parquet")):
    raise RuntimeError("no matching change was written: check the producer and Kafka, then run the queries again")
found = spark.read.parquet(str(MATCHES_PATH))
print("matching changes:", found.count())

found.groupBy("entity_type").agg(F.count("*").alias("changes"),
                                 F.countDistinct("imdb_id").alias("entities")).show()

(found.groupBy("entity_type", "imdb_id", "imdb_name", "imdb_type", "imdb_year")
 .agg(F.count("*").alias("changes"), F.countDistinct("user").alias("editors"),
      F.sum("bytes_changed").alias("bytes_changed"))
 .orderBy(F.desc("changes")).show(20, truncate=False))

found.orderBy("event_time").select("event_time", "title", "imdb_name", "imdb_id", "user", "bot", "bytes_changed") \
    .show(20, truncate=40)

found.drop("comment").toPandas().to_csv(RESULTS_DIR / "matches.csv", index=False)

Final value of each window: the version written by the last micro-batch.

In [ ]:
versions = spark.read.parquet(str(WINDOWS_PATH))
last = Window.partitionBy("window_start", "imdb_id").orderBy(F.desc("batch_id"))
final_windows = versions.withColumn("r", F.row_number().over(last)).filter("r = 1").drop("r", "batch_id")

(final_windows.groupBy("window_start", "window_end")
 .agg(F.sum("changes").alias("changes"), F.countDistinct("imdb_id").alias("entities"))
 .orderBy("window_start").show(truncate=False))
final_windows.toPandas().to_csv(RESULTS_DIR / "window_counts.csv", index=False)

### Streaming metrics

In [ ]:
def progress_table(query):
    rows = []
    for p in query.recentProgress:
        state = p.get("stateOperators", [])
        rows.append({
            "query": p["name"], "batch_id": p["batchId"], "timestamp": p["timestamp"],
            "input_rows": p["numInputRows"],
            "input_rows_per_s": round(p.get("inputRowsPerSecond", 0.0), 2),
            "processed_rows_per_s": round(p.get("processedRowsPerSecond", 0.0), 2),
            "batch_duration_ms": p["durationMs"].get("triggerExecution"),
            "watermark": p.get("eventTime", {}).get("watermark"),
            "state_rows": sum(s.get("numRowsTotal", 0) for s in state),
            "dropped_by_watermark": sum(s.get("numRowsDroppedByWatermark", 0) for s in state),
        })
    return pd.DataFrame(rows)


metrics = pd.concat([progress_table(q_matches), progress_table(q_windows)], ignore_index=True)
metrics.to_csv(RESULTS_DIR / "progress.csv", index=False)
print(metrics.groupby("query")[["input_rows", "dropped_by_watermark"]].sum())
metrics.tail(10)

## 3. The `REPORT.md` should include an explanation of your chosen method to deal with late arriving data.

Numbers for the report: delay between the change on Wikipedia and its arrival in Kafka. The producer holds back a small share of the events on purpose (`--late-ratio`, `--late-minutes`) to check what the watermark does with them.

In [ ]:
(found.withColumn("simulated", F.col("simulated_delay_s").isNotNull())
 .groupBy("simulated")
 .agg(F.count("*").alias("matched_changes"),
      F.round(F.avg("delay_seconds"), 1).alias("avg_delay_s"),
      F.expr("percentile(delay_seconds, 0.5)").alias("median_delay_s"),
      F.max("delay_seconds").alias("max_delay_s"))
 .show())

late_summary = metrics[metrics["query"] == "window_counts"][["batch_id", "watermark", "dropped_by_watermark"]]
late_summary.to_csv(RESULTS_DIR / "late_data.csv", index=False)
print("rows dropped by the watermark in the window aggregation:", late_summary["dropped_by_watermark"].sum())